# Phase 2: Tools and ReAct (Routing Agent)

An agent can call tools. In a ReAct loop it reasons, acts (calls a tool), reads the result, and repeats until it can answer.

Our Routing Agent has two tools and must chain them: area to municipality, then municipality to department.

In [ ]:
import json
from pathlib import Path

import community_voice
from community_voice.agents.routing import run_routing_agent
from community_voice.agents.triage import triage
from community_voice.schemas.report import ReportIn, SampleReport
from community_voice.schemas.triage import TriageResult
from community_voice.tools.directory import find_municipality, lookup_department

ROOT = Path(community_voice.__file__).resolve().parents[2]

## 1. The tools

Plain Python functions with `@tool`. The docstring tells the agent when to use each one. 

In [ ]:
print(find_municipality.invoke({"area": "Umlazi"}))
print(lookup_department.invoke({"municipality": "eThekwini", "category": "water"}))

## 2. Route one report

Triage first, then route. `show_trace` prints each tool call (ACT) and what came back (RESULT).

In [ ]:
def show_trace(state):
    for message in state["messages"]:
        if message.type == "ai" and message.tool_calls:
            for call in message.tool_calls:
                if call["name"] != "RoutingResult":
                    print("ACT   ", call["name"], call["args"])
        elif message.type == "tool" and message.name != "RoutingResult":
            print("RESULT", message.content)

In [ ]:
report = ReportIn(
    text="Amanzi ayaphuma emgwaqeni e-Umlazi V Section, "
    "kuvuza ipayipi elikhulu kusukela izolo."
)
triaged = triage(report)
state = run_routing_agent(triaged)
show_trace(state)
print(state["structured_response"].model_dump_json(indent=2))

## 3. Route all 10 sample reports

In [ ]:
path = ROOT / "data" / "sample_reports" / "reports.json"
samples = [
    SampleReport.model_validate(item)
    for item in json.loads(path.read_text(encoding="utf-8"))
]
for sample in samples:
    triaged = triage(ReportIn(text=sample.text))
    routing = run_routing_agent(triaged)["structured_response"]
    print(
        f"{sample.id}  {triaged.category.value:11}"
        f" {str(triaged.area_mentioned):17} {routing.municipality:19}"
        f" {routing.department:22} human: {routing.needs_human}"
    )

## 4. An area that is not in the directory

The agent must not guess. It should answer `unknown` and ask for a human.

In [ ]:
unknown = TriageResult(
    language="en",
    english_text="No water since Monday.",
    category="water",
    summary="Water outage.",
    urgency="high",
    urgency_reason="Outage for days.",
    area_mentioned="Thornville Heights",
)
state = run_routing_agent(unknown)
show_trace(state)
print(state["structured_response"].model_dump_json(indent=2))

## What to notice

- The agent chose which tool to call, and in what order.
- It used the first tool's result as input to the second.
- With no match, it asked for a human instead of guessing.
- To use Gemini instead of Groq, pass `provider='gemini'` to `run_routing_agent`.